# Mall Customer Segmentation - Unsupervised Learning Capstone

Dataset: [Mall Customer Segmentation Data](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python) (Kaggle, `vjchoudhary7`). 200 rows, 5 columns - see README.md for download steps and the network-access caveat for this scaffold.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from src.clustering_utils import (
    load_raw, scale_features, scan_k, fit_kmeans, profile_clusters, FEATURE_COLUMNS
)

sns.set_theme(style="whitegrid")

## Milestone 1 - Load & scale

Annual Income (tens of thousands) and Spending Score (1-100) are on very different scales - unscaled K-Means would let income dominate the distance metric almost entirely.

In [ ]:
df = load_raw()
print(df.shape)
df.head()

In [ ]:
X_scaled, scaler = scale_features(df)
print("Scaled feature means (should be ~0):", X_scaled.mean(axis=0).round(3))
print("Scaled feature stds (should be ~1):", X_scaled.std(axis=0).round(3))

## Milestone 2 - Choosing k: elbow method + silhouette score

In [ ]:
ks, inertias, sil_scores = scan_k(X_scaled, k_range=range(2, 11))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(ks, inertias, "o-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia (WCSS)"); axes[0].set_title("Elbow method")
axes[1].plot(ks, sil_scores, "o-", color="green")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score"); axes[1].set_title("Silhouette vs k")
fig.tight_layout()
plt.show()

**Chosen k and justification:** _(TODO(you) - pick k based on BOTH plots, not just "the elbow looks like...". State the silhouette score at your chosen k.)_

In [ ]:
CHOSEN_K = 5  # TODO(you): replace with your justified choice
km = fit_kmeans(X_scaled, k=CHOSEN_K)
print(f"KMeans silhouette at k={CHOSEN_K}: {silhouette_score(X_scaled, km.labels_):.3f}")

## Milestone 3 - Comparison method: Hierarchical clustering

In [ ]:
linked = linkage(X_scaled, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(linked, truncate_mode="lastp", p=30)
plt.title("Hierarchical clustering dendrogram (truncated)")
plt.xlabel("Cluster size"); plt.ylabel("Distance")
plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=CHOSEN_K, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)
print(f"Agglomerative silhouette at k={CHOSEN_K}: {silhouette_score(X_scaled, agg_labels):.3f}")
print(f"KMeans silhouette at k={CHOSEN_K}:        {silhouette_score(X_scaled, km.labels_):.3f}")

## Milestone 4 - PCA visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"Explained variance ratio: {pca.explained_variance_ratio_.round(3)} "
      f"(total: {pca.explained_variance_ratio_.sum():.3f})")

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=km.labels_, cmap="tab10", alpha=0.7)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"KMeans clusters (k={CHOSEN_K}) - PCA projection")
plt.colorbar(scatter, label="cluster")
plt.show()

## Milestone 5 - Cluster interpretation

In [ ]:
profile = profile_clusters(df, km.labels_)
profile

**Persona names:** _(TODO(you) - for each cluster row above, write a one-sentence business persona, e.g. "high income, low spending - potential upsell target".)_

- Cluster 0: 
- Cluster 1: 
- Cluster 2: 

## Milestone 6 - Honest validation

State your final silhouette score plainly. For a dataset this small and this loosely structured, scores above ~0.5–0.6 are already good - do not overclaim cluster quality.